# ipyaidojo

> Start ipyai with a worked tooling session already in its history

In [ ]:
#| default_exp ipyaidojo

#| export
An ipyai session is an aidialog `Dialog`, saved whole as an `.ipynb` under `./.ipyai/sessions/`, and ipyai rebuilds the model's context from that dialog on every turn (`dlg2hist`). So the template needs no host-native record format: the packaged dialogs already have the shape ipyai resumes. What differs from the clikernel hosts is the tool and the bootstrap. ipyai's model runs code through `py`, which runs the cell in the user's kernel and renders its outputs with the same `render_text` clikernel uses, so the baked round must show `py` calls; `ipyaidojo` gets them by replaying the cells through ipyai's own kernel client, with no model spend. And ipyai has no `clik` skill to document, so its bootstrap is its own dialog, captured from an ipyai session, assembled with the shared round at build time. Launch is then a session write plus `ipyai -r`.

In [ ]:
#| export
import asyncio, json, os, sys, tempfile
from importlib.resources import files
from fastcore.utils import *
from fastcore.script import call_parse
from aidialog.ipynb import read_ipynb, write_ipynb, reads_ipynb
from aidialog.msg_parts import ToolResponse, tool_text
from llmdojo.tmpl import save_store, load_store, load_reg, find_cid, launch_config, assemble, localized_cells, refresh_dialog, boot_gates
from llmdojo.tmpl import _seed_doced, _pmsgs, _msg_cells, _one_prompt

In [ ]:
from fastcore.test import *
import shutil, ipyai.config as ipyai_config
from ipyai.kernel import KernelSession
from ipyai.bridge import setup_tools
from ipyai.session import resolve_session
from llmdojo.tmpl import TMPL_PROMPT
from aidialog.hist import reply2dlg, _parse_call

## The ipyai round

The clikernel hosts' bootstrap opens with their own skill, `doc(clik, pysk, edsk)`, because their startup banner says so. ipyai's model never sees clikernel and gets no banner, so its bootstrap prompt names the reads itself, and its bootstrap dialog is captured from an ipyai session that answered it (`capture_boot`, below). The round is the shared one, every cell unchanged.

In [ ]:
#| export
TOOL = 'py'   # ipyai's model-facing tool: runs a cell in the user's kernel
BOOT_PATH = files('llmdojo')/'dojo_data'/'ipyai_boot.ipynb'   # ipyai's bootstrap: one prompt, its reply the startup's doc() reads through `py`
BOOT_PROMPT = "Bootstrap with py, one call each: doc(pysk, edsk), then doc(dsk, exh, rgsk), then doc(acp), then list_pyskills(). Then tell me when you're ready."

## Replaying through ipyai

`replay_cells` plays cells the way a live ipyai session's model does: a fresh kernel owned by ipyai's `KernelSession`, seeded by `setup_tools` (which also runs the user's `~/.config/ipyai/startup.py`, so the round's imports are the real ones, just as clikernel's replay runs the clikernel startup), and each cell through the `py` tool. The result is the text the model would read back, with media tags where a cell displayed an image.

In [ ]:
#| export
async def replay_cells(
    cells, # Kernel cell sources, run in order
    cwd, # Directory to start the kernel in
    env=None, # Extra environment entries for the kernel process
):
    "Each cell's `py` result from a fresh ipyai kernel at `cwd`: the text the model reads in a live session"
    from ipyai.kernel import KernelSession
    from ipyai.bridge import setup_tools
    k = await KernelSession().start(cwd=cwd, env=env)
    try:
        _,tools = await setup_tools(k.kc)
        outs = []
        for c in cells:
            r = await tools.call_text(TOOL, dict(code=c))
            outs.append(tool_text(r.content if isinstance(r, ToolResponse) else r))
        return outs
    finally: await k.close()

A bare expression comes back as its value, a printed line as its text with the newline it printed, and a silent cell as nothing at all, exactly as `py` renders them live (this needs a running rustygate):

In [ ]:
rd = Path(tempfile.mkdtemp())
outs = await replay_cells(['1+1', "print('hi')", 'x = 3'], rd)
test_eq(outs, ['2', 'hi\n', ''])
outs

## The template store

`ipyai_dialog` assembles ipyai's bootstrap with the shared round and makes it ipyai's: every baked call is parsed back to its cell, the cells are localized to the real run dir and replayed through `py`, each call is rewritten as a `py` call with its fresh result, the bootstrap and the round are gated, and the run dir is canonicalized again so the stored artifact reads the same everywhere. The store holds that dialog as one notebook dict, the completion id the replay earned, and the names the template documents.

In [ ]:
#| export
TMPL_DIR = files('llmdojo')/'dojo_data'/'ipyai_store'   # package data: compiled by dojobuild, shipped with the code

async def ipyai_dialog(
    boot=None, # ipyai's bootstrap dialog (or its path); the packaged one if None
    rnd=None, # The round dialog (or its path); the packaged round if None
    cwd=None, # Directory to replay in; the current directory if None
):
    "The template as ipyai plays it: its bootstrap and the shared round, calls renamed to `py`, results regenerated by replay, gated, canonicalized; returns `(dialog, outs)`"
    dlg = assemble(boot or BOOT_PATH, rnd)
    outs = [await replay_cells(cells, cwd or '.') for cells in localized_cells(dlg)]
    return refresh_dialog(dlg, outs, tool=TOOL), outs

async def build_template(
    boot=None, # ipyai's bootstrap dialog (or its path); the packaged one if None
    rnd=None, # The round dialog (or its path); the packaged round if None
    d=None, # Store dir; `TMPL_DIR` if None
):
    "Replay the template through ipyai and write the ipyai store: the dialog as a notebook dict, its completion id, and its doced list"
    with tempfile.TemporaryDirectory(prefix='ipyaidojo_') as td: dlg,outs = await ipyai_dialog(boot, rnd, td)
    save_store(Path(d or TMPL_DIR), [json.loads(write_ipynb(dlg))], find_cid(L(outs).concat()), doced=dlg.meta['llmdojo']['doced'])
    return dlg

def load_template(
    d=None, # Store dir; `TMPL_DIR` if None
):
    "The stored template items and metadata"
    return load_store(Path(d or TMPL_DIR))

def template_dialog(
    items, # Stored template items: one notebook dict
):
    "The template dialog read back from its stored notebook dict"
    return reads_ipynb(json.dumps(items[0]))

Building from the packaged dialog replays the whole round (about a minute, no model spend), here with the llmdojo state redirected to the store dir so the replay's receipts stay out of the real registry and a parallel notebook run cannot share its run dir. The result opens with the swapped bootstrap read and calls `py` throughout, and the store reads back to the same reply:

In [ ]:
tstore = Path(tempfile.mkdtemp())
os.environ['LLMDOJO_STATE_DIR'] = str(tstore)
try: tdlg = await build_template(d=tstore)
finally: del os.environ['LLMDOJO_STATE_DIR']
test_eq([m.content for m in tdlg.messages], [BOOT_PROMPT, TMPL_PROMPT])
calls = [_parse_call(m.content) for pm in tdlg.messages for m in reply2dlg(pm).messages if m.msg_type=='code']
test_eq({c[0] for c in calls}, {TOOL})
assert 'clik' not in tdlg.meta['llmdojo']['doced']
items,meta = load_template(tstore)
test_eq(template_dialog(items).messages[1].ai_res, tdlg.messages[1].ai_res)
meta

## Capturing the bootstrap

ipyai's bootstrap is played live, in ipyai: a fresh session in a Python project answers `BOOT_PROMPT` with the doc reads through `py`, and `capture_boot` takes that reply from the session file. The session is already a dialog in the template's shape, so the capture is a selection, not a conversion: the last prompt whose reply passes `boot_gates` (nothing but doc reads and catalog calls), with the prompt text set to the canonical `BOOT_PROMPT`, written as a one-prompt dialog for review and for copying over the packaged `ipyai_boot.ipynb`.

In [ ]:
#| export
async def capture_boot(
    sess=None, # An ipyai session id or name prefix; the project's newest session if None
    cwd=None, # Project directory; the current directory if None
    d=None, # Template store dir; `TMPL_DIR` if None
):
    "ipyai's bootstrap from a session that answered `BOOT_PROMPT`: the last prompt whose reply is nothing but doc reads, replayed for full results, written to `boot.ipynb` beside the store"
    from ipyai.session import resolve_session, list_sessions
    path = resolve_session(sess, cwd or '.') if sess else list_sessions(cwd or '.')[0][0]
    dlg = read_ipynb(str(path))
    pm = last(m for m in _pmsgs(dlg) if not boot_gates(_msg_cells(m)[2]))
    if pm is None: raise ValueError(f'no bootstrap reply (a prompt answered only with doc reads) in {path}')
    b = _one_prompt(pm, 'ipyai_boot')
    b.messages[0].content = BOOT_PROMPT
    with tempfile.TemporaryDirectory(prefix='ipyaidojo_') as td: outs = [await replay_cells(cells, td) for cells in localized_cells(b)]
    refresh_dialog(b, outs, tool=TOOL)   # the session file stores tool results truncated; the replay restores the full text the model read
    out = Path(d or TMPL_DIR)/'boot.ipynb'
    write_ipynb(b, str(out))
    return out

## Launching

`prep_dojo` writes the template as a session of the target project through ipyai's own `Session` (so the `.ipyai/sessions/` layout and its self-ignoring `.gitignore` are ipyai's), registers the completion id, and returns the session id that `ipyai -r` resumes. Resuming paints the round and hands the dialog to the model as real `py` tool calls.

In [ ]:
#| export
def prep_dojo(
    cwd=None, # Project to start in; the current directory if None
    d=None, # Template store dir; `TMPL_DIR` if None
):
    "Write the template as a session of `cwd` through ipyai's `Session`, register its completion id, seed its doc-state, and return the session id to resume"
    from ipyai.session import Session
    items,meta = load_reg(d, TMPL_DIR, 'ipyaidojo')
    s = Session(root=cwd or '.')
    s.save(template_dialog(items))
    if ds := meta.get('doced'): _seed_doced(s.path.stem, ds)   # ipyai names the session stem in the kernel's LLMDOJO_HOST_ID
    return s.path.stem

Against temporary dirs, with the llmdojo state redirected so the registration stays off the real completion record:

In [ ]:
mproj = Path(tempfile.mkdtemp())
os.environ['LLMDOJO_STATE_DIR'] = str(tstore)
try: psid = prep_dojo(mproj, tstore)
finally: del os.environ['LLMDOJO_STATE_DIR']
back = read_ipynb(resolve_session(psid, mproj))
test_eq([m.content for m in back.messages], [BOOT_PROMPT, TMPL_PROMPT])
test_eq(back.messages[1].ai_res, tdlg.messages[1].ai_res)
psid

The launcher is a `call_parse` CLI: `--sid` prints the prepared id instead of launching, `--capture` takes ipyai's bootstrap from the project's newest session, unrecognized flags are forwarded to `ipyai`, and standing arguments come from the `ipyai_args` list in `$XDG_CONFIG_HOME/ipyaidojo/config.toml` through `launch_config('ipyai')`.

In [ ]:
#| export
@call_parse(nested=True)
def main(
    sid:bool=False, # Print the prepared session id instead of launching ipyai
    capture:bool=False, # Capture ipyai's bootstrap from this project's newest session into the store's `boot.ipynb`, and exit
):
    "Prepare an ipyai session opening with the worked round and launch `ipyai -r` on it; template maintenance is `dojobuild`"
    if capture: return print(f"wrote: {asyncio.run(capture_boot())}")
    s = prep_dojo()
    if sid: return print(s)
    os.execvp('ipyai', ['ipyai', *launch_config('ipyai'), '-r', s, *sys.argv[1:]])

## Rules in ipyai kernels

llmdojo's `ipyai/startup.py` (linked into `~/.config/ipyai/`) also installs the session rules, and applies them to the model's cells only: ipyai runs the model's `py` cells with `store_history=False` and the user's typed cells with `True`, so IPython's `pre_run_cell` info tells them apart. A blocking rule rejects the model's cell and the rejection reaches the model as the tool result; the same cell typed by the user runs untouched. Here against the checkout's startup file, with the kernel's llmdojo state kept in a temp dir:

In [ ]:
ipyai_config.STARTUP_PATH = Path(files('llmdojo')).parent/'ipyai'/'startup.py'
rd2 = Path(tempfile.mkdtemp())
k = await KernelSession().start(cwd=rd2, env=dict(LLMDOJO_STATE_DIR=str(rd2)))
try:
    _,tools = await setup_tools(k.kc)
    blocked = await tools.call_text(TOOL, dict(code='!echo hi'))
    user = await k.kc.run('!echo hi')
finally: await k.close()
assert 'RuleBlock' in blocked and 'Bash tool' in blocked
test_eq(user[0]['text'].strip(), 'hi')
blocked.splitlines()[-1]

## Cleanup

In [ ]:
for p in (rd, rd2, tstore, mproj): shutil.rmtree(p)

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()